# RAG Workshop: From Zero to Production

**Build a Retrieval-Augmented Generation system from scratch**

Tools: LangChain + OpenAI + ChromaDB

---

This notebook covers all workshop examples in one place.
Run cells top-to-bottom. Each section builds on the previous one.

> **Google Colab users:** Click `Runtime → Run all` or run cells one by one.

## Setup

Install dependencies and configure your API key.

In [1]:
# Install all dependencies (takes ~2 minutes on Colab)
! pip install -q langchain langchain-openai langchain-community \
    chromadb openai pypdf tiktoken ragas datasets \
    sentence-transformers rank_bm25 langchain-cohere langchain_chroma \
    langgraph

In [2]:
import os

# Set your OpenAI API key
# Option 1: Paste directly (for quick demos only)
# os.environ['OPENAI_API_KEY'] = 'sk-your-key-here'

# Option 2: Use Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('API key loaded from Colab Secrets')
except:
    if not os.getenv('OPENAI_API_KEY'):
        print('WARNING: Set OPENAI_API_KEY above!')

In [3]:
# Download the sample PDF (Attention Is All You Need)
! curl -L -o attention_is_all_you_need.pdf https://arxiv.org/pdf/1706.03762
print('PDF downloaded successfully!')

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2163k  100 2163k    0     0  4394k      0 --:--:-- --:--:-- --:--:-- 4388k
PDF downloaded successfully!


In [4]:
import warnings
warnings.filterwarnings('ignore')

---
## Module 1: Foundations

### What is RAG?

**RAG = Give the LLM the right context at question time, not training time.**

```
User Question → [Find Relevant Docs] → [LLM reads docs + Question] → Answer
```

### Why RAG?

| Problem | RAG Solution |
|---------|-------------|
| Knowledge cutoff | Retrieve up-to-date documents |
| Hallucinations | Ground answers in real sources |
| No private data | Search your own documents |
| Context limits | Retrieve only relevant chunks |

---
## Module 2: Build Your First RAG System

### Step 1: Load the Document

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('FAQ.pdf')
pages = loader.load()

print(f'Loaded {len(pages)} pages')
print(f'\nFirst 500 chars of page 1:\n')
print(pages[0].page_content[:500])
print(f'\nMetadata: {pages[0].metadata}')

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loaded 3 pages

First 500 chars of page 1:

Frequently Asked 
Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 1 
 1. How will the OHS Regulations affect my business?
The Regulations will impact you 
and your business on various 
levels.
With fewer approvals required 
from the Chief Safety Officer, the 
Regulations strongly support an 
Internal Responsibility System (IRS).
 2. How different are the OHS Regulations from the General Safety Regulations?
There are many changes 
throughout the Regulations. 
The document is easier to u

Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260410122842', 'source': 'FAQ.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}


### Step 2: Chunk the Documents

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=['\n\n', '\n', '.', ' ', '']
)

chunks = splitter.split_documents(pages)

print(f'Original pages: {len(pages)}')
print(f'After chunking: {len(chunks)} chunks')
print(f'\nSample chunk:\n{chunks[3].page_content}')

Original pages: 3
After chunking: 5 chunks

Sample chunk:
regulations have Codes. The WSCC 
will update Codes to reflect the 
Regulations, but it is important that 
you are familiar with, and ready to 
comply with, the Regulations. 
  5. Is there a grace period? How much time do I have to become compliant  
with the OHS Regulations? 
 7. As a worker, do my responsibilities change under the OHS Regulations?
Not really. 
Every worker has a responsibility to 
work safely. These responsibilities 
are currently part of the Safety 
Acts. They will now be part of the 
Regulations as well.
All workers must use safeguards, 
safety equipment, and personal 
protective equipment required by 
the Regulations. They must also 
follow safe work practices and 
procedures developed under these  
Regulations.


### Step 3: Embed and Store in ChromaDB

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Quick demo:
sample = embeddings.embed_query('How will the OHS Regulations affect my business?')
print(f'Embedding dimensions: {len(sample)}')
print(f'First 5 values: {sample[:5]}')

# Create vector store (may take 30-60 seconds)
print(f'\nEmbedding {len(chunks)} chunks...')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_db'
)
print(f'Vector store created with {vectorstore._collection.count()} vectors')

collections = vectorstore._client.list_collections()
for col in collections:
    print(col.name)

Embedding dimensions: 1536
First 5 values: [-0.019989013671875, 0.02020263671875, 0.058685302734375, 0.01043701171875, -0.049896240234375]

Embedding 5 chunks...
Vector store created with 5 vectors
langchain


### Demo: Similarity Search

In [8]:
query = 'How will the OHS Regulations affect my business?'
results = vectorstore.similarity_search_with_score(query, k=3)

print(f'Top 3 results for: "{query}"\n')
for i, (doc, score) in enumerate(results):
    print(f'--- Result {i+1} (distance: {score:.4f}) ---')
    print(f'Page {doc.metadata.get("page", "?")}: {doc.page_content[:200]}...\n')

Top 3 results for: "How will the OHS Regulations affect my business?"

--- Result 1 (distance: 0.4787) ---
Page 0: Frequently Asked 
Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 1 
 1. How will the OHS Regulations affect my business?
The Regulations will impact you 
and your business on various 
level...

--- Result 2 (distance: 0.6866) ---
Page 1: Frequently Asked Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 2 
  4. How much will it cost me to comply with the OHS Regulations?
It depends.
Complying with the OHS Regulations 
is a pro...

--- Result 3 (distance: 0.6998) ---
Page 1: regulations have Codes. The WSCC 
will update Codes to reflect the 
Regulations, but it is important that 
you are familiar with, and ready to 
comply with, the Regulations. 
  5. Is there a grace per...



### Step 4: Build the RAG Chain

In [9]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(model='gpt-5')

rag_prompt = PromptTemplate(
    template="""You are a helpful assistant. Use ONLY the context below to answer.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:""",
    input_variables=['context', 'question']
)

retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 4})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    chain_type_kwargs={'prompt': rag_prompt},
    return_source_documents=True
)

print('RAG chain ready!')

RAG chain ready!


### Step 5: Ask Questions!

In [10]:
def ask(question):
    result = qa_chain.invoke({'query': question})
    print(f'Q: {question}')
    print(f'A: {result["result"]}')
    print(f'Sources: {[d.metadata.get("page","?") for d in result["source_documents"]]}')
    print()

ask('How will the OHS Regulations affect my business?')
ask('What is the easiest way to find out how to be compliant? ')
ask('Who invented the iPhone?')  # Should say 'not enough information'

Q: How will the OHS Regulations affect my business?
A: The Regulations will impact you and your business on various levels. With fewer approvals required from the Chief Safety Officer, the Regulations strongly support an Internal Responsibility System (IRS).
Sources: [0, 1, 1, 2]

Q: What is the easiest way to find out how to be compliant? 
A: Start with the Table of Contents—it clearly outlines what you need and is available at OHSregs.ca. If in doubt, call the WSCC at 1-800-661-0792 (Northwest Territories) or 1-877-404-4407 (Nunavut) and ask to speak to a Safety Officer.
Sources: [1, 0, 1, 2]

Q: Who invented the iPhone?
A: I don't have enough information.
Sources: [0, 2, 1, 0]



---
## RAG vs No-RAG Comparison

The most compelling demo — see hallucination prevention in action.

In [11]:
from langchain.schema import HumanMessage, SystemMessage

def ask_without_rag(question):
    messages = [
        SystemMessage(content='Answer directly and concisely.'),
        HumanMessage(content=question)
    ]
    return llm.invoke(messages).content

question = 'Who is going to approve the complaint?'

print('WITHOUT RAG (pure LLM):')
print(ask_without_rag(question))

print('\n' + '─' * 50)

print('\nWITH RAG (retrieved + grounded):')
result = qa_chain.invoke({'query': question})
print(result['result'])
print(f'\nSources: pages {[d.metadata.get("page","?") for d in result["source_documents"]]}')

WITHOUT RAG (pure LLM):
It depends on the context.

- Court case: a judge or magistrate (the clerk only files it; prosecutors authorize criminal complaints).
- Workplace grievance: HR or the designated grievance officer/manager.
- Online/platform: site moderators or admins.
- Government agency: the assigned case officer/supervisor.

What kind of complaint and where was it filed?

──────────────────────────────────────────────────

WITH RAG (retrieved + grounded):
I don't have enough information.

Sources: pages [1, 2, 1, 0]


---
## Module 3: Advanced RAG Techniques

---
## Re-ranking with Cohere

Improve retrieval precision by first fetching a large number of documents (e.g., 20) with basic semantic search, then using a powerful cross-encoder to re-score and select the top few.

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


loader = PyPDFLoader('attention_is_all_you_need.pdf')
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=['\n\n', '\n', '.', ' ', '']
)

chunks = splitter.split_documents(pages)

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_db'
)

In [13]:
from dotenv import load_dotenv
import os

from langchain_cohere import CohereRerank
from langchain.retrievers import ContextualCompressionRetriever

load_dotenv()

base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

compressor = CohereRerank(
    model="rerank-english-v3.0",
    cohere_api_key=os.getenv("COHERE_API_KEY"),
    top_n=3,
)

retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

query = "Explain attention mechanism in transformers?"
reranked_docs = retriever.invoke(query)

for doc in reranked_docs:
    print(doc.metadata.get("relevance_score"), doc.page_content)
    print("************************************")

0.99683964 The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
************************************
0.9939261 sequential nature precludes 

---

### Multi-Query Retrieval

Generate multiple query variations to capture more relevant chunks.

In [14]:
import logging
from langchain.retrievers.multi_query import MultiQueryRetriever

logging.basicConfig()
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k': 3}),
    llm=ChatOpenAI(model='gpt-5')
)

query = 'What is the easiest way to find out how to be compliant?'
docs = multi_retriever.invoke(query)
print(f'\nRetrieved {len(docs)} unique chunks with multi-query')
for i, doc in enumerate(docs):
    print(f'  [{i+1}] Page {doc.metadata.get("page","?")}: {doc.page_content[:120]}...')

INFO:langchain.retrievers.multi_query:Generated queries: ['How can I quickly identify the compliance requirements (laws, regulations, policies) that apply to me or my organization?', 'Is there a simple guide or checklist to determine what I need to do to be compliant with relevant standards and regulations?', 'What are the easiest steps and resources (e.g., self-assessment, gap analysis, official guidance) to understand and confirm compliance?']



Retrieved 4 unique chunks with multi-query
  [1] Page 1: Frequently Asked Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 2 
  4. How much will it cost me to comply wi...
  [2] Page 0: Frequently Asked 
Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 1 
 1. How will the OHS Regulations affect m...
  [3] Page 2: Frequently Asked Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 3 
Ask Us. 
It is important that you understa...
  [4] Page 1: regulations have Codes. The WSCC 
will update Codes to reflect the 
Regulations, but it is important that 
you are famil...


---

### Hybrid Search

BM25 keyword + semantic search combined

In [15]:
from dotenv import load_dotenv
import os

from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

# ---- OpenAI embeddings ----
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# ---- Load existing Chroma collection ----
vectorstore = Chroma(
    collection_name="langchain",
    persist_directory="./chroma_db",   # same path used during ingestion
    embedding_function=embeddings,
)

# ---- Semantic retriever from existing Chroma DB ----
vector_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6},
)

# ---- Build BM25 over already-ingested documents pulled from Chroma ----
stored = vectorstore.get(include=["documents", "metadatas"])

documents = stored["documents"]
metadatas = stored.get("metadatas", [{} for _ in documents])

from langchain_core.documents import Document
bm25_docs = [
    Document(page_content=doc, metadata=meta or {})
    for doc, meta in zip(documents, metadatas)
]

bm25_retriever = BM25Retriever.from_documents(bm25_docs)
bm25_retriever.k = 6

# ---- Hybrid retriever ----
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)

# ---- Query ----
query = "Who approves the OHS regulations?"
results = hybrid_retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")
    print(doc.metadata)
    print("-" * 80)

1. Frequently Asked 
Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 1 
 1. How will the OHS Regulations affect my business?
The Regulations will impact you 
and your business on various 
levels.
With fewer approvals required 
from the Chief Safety Officer, the 
Regulations strongly support an 
Internal Responsibility System (IRS).
 2. How different are the OHS Regulations from the General Safety Regulations?
There are many changes 
throughout the Regulations. 
The document is easier to use with 
an indexed search feature to help 
you find the information you need.
Refer to the Table of Contents in the 
Draft document and review relevant 
sections to know what, if any, 
changes you must make.
New requirements in the 
Regulations address:
• Hazard Assessment
• Supervisor Education
• Robotics
• Radiation
• Explosives
• Forestry and Mill Operations
• Additional protection for electrical 
workers, healthcare workers, and 
firefighters
Other changes address:
• OHS Committees
{'pag

### Conversational RAG with Memory

In [16]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    k=5, memory_key='chat_history', return_messages=True, output_key='answer'
)

conv_chain = ConversationalRetrievalChain.from_llm(
    llm=ChatOpenAI(model='gpt-4o-mini', temperature=0),
    retriever=vectorstore.as_retriever(search_kwargs={'k': 4}),
    memory=memory,
    return_source_documents=True
)

def chat(msg):
    result = conv_chain.invoke({'question': msg})
    print(f'You: {msg}')
    print(f'AI:  {result["answer"]}\n')

chat('How will the OHS Regulations affect my business?')
chat('Who is going to approve it?')  # 'it' → Transformer
chat('How much will it cost me to comply with the OHS Regulations?')  # builds on context

/var/folders/9f/hg5k6b2d4w11chb41n0fkf540000gp/T/ipykernel_93426/4035434939.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(


You: How will the OHS Regulations affect my business?
AI:  The OHS Regulations will impact your business on various levels. With fewer approvals required from the Chief Safety Officer, the Regulations strongly support an Internal Responsibility System (IRS). This means that your business will need to adapt to the new requirements and ensure compliance with the updated safety standards.

You: Who is going to approve it?
AI:  The context does not specify who will be responsible for the approval under the OHS Regulations. It only mentions that there will be fewer approvals required from the Chief Safety Officer.

You: How much will it cost me to comply with the OHS Regulations?
AI:  It depends. Complying with the OHS Regulations is a progressive step in your safety program. Costs you may incur to become compliant include training and education, and potential equipment updates, such as fall protection. You will also need to allocate time for safety committees, supervisor training, proper w

### RAG Evaluation with RAGAS

Measure quality with Faithfulness, Answer Relevancy, and Context Precision.

In [17]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall, Faithfulness,AnswerRelevancy

# Setup LLM
client = AsyncOpenAI()
llm = llm_factory("gpt-5", client=client)
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)


# Create metric
scorer_context_precision = ContextPrecision(llm=llm)
scorer_context_recall = ContextRecall(llm=llm)
scorer_faithfulness = Faithfulness(llm=llm)
scorer_answer_relevancy = AnswerRelevancy(llm=llm, embeddings=embeddings)


user_input="How will the OHS Regulations affect my business?"
reference="With fewer approvals required from the Chief Safety Officer, the Regulations strongly support an Internal Responsibility System (IRS)."
retrieved_contexts=[
        "With fewer approvals required from the Chief Safety Officer, the Regulations strongly support an Internal Responsibility System (IRS)."
    ]

# Evaluate
result = await scorer_context_precision.ascore(
    user_input=user_input,
    reference=reference,
    retrieved_contexts=retrieved_contexts
)
print(f"Context Precision Score: {result.value}")

result = await scorer_context_recall.ascore(
    user_input=user_input,
    reference=reference,
    retrieved_contexts=retrieved_contexts
)
print(f"Context Recall Score: {result.value}")

# result = await scorer_faithfulness.ascore(
#     user_input=user_input,
#     response=reference,
#     retrieved_contexts=retrieved_contexts
# )
# print(f"Faithfulness Score: {result.value}")

result = await scorer_answer_relevancy.ascore(
    user_input=user_input,
    response=reference
)
print(f"Answer Relevancy Score: {result.value}")

Context Precision Score: 0.9999999999
Context Recall Score: 1.0
Answer Relevancy Score: 0.41071020315368806


---
## Agentic RAG

The LLM as an intelligent orchestrator

In [18]:
from dotenv import load_dotenv
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.tools.retriever import create_retriever_tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY"),
)

vectorstore = Chroma(
    collection_name="langchain",
    persist_directory="./chroma_db",
    embedding_function=embeddings,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retriever_tool = create_retriever_tool(
    retriever,
    "search_documents",
    "Search company documents for relevant information before answering.",
)

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
)

agent = create_react_agent(
    llm,
    tools=[retriever_tool],
)

query = """
Who approves the OHS regulations?
"""

result = agent.invoke(
    {
        "messages": [
            SystemMessage(
                content=(
                    "You are a document QA assistant. "
                    "Use the search_documents tool at most once if needed. "
                    "If the retrieved context is sufficient, answer directly. "
                    "Do not call the same tool repeatedly for the same question. "
                    "After using the tool, provide the final answer."
                )
            ),
            HumanMessage(content=query),
        ]
    },
    config={"recursion_limit": 50},
)

print(result["messages"][-1].content)

The OHS (Occupational Health and Safety) Regulations are approved with fewer approvals required from the Chief Safety Officer, supporting an Internal Responsibility System (IRS). The Workers' Safety and Compensation Commission (WSCC) is involved in updating Codes to reflect the Regulations and ensuring compliance.


In [19]:
for step in agent.stream(
    {"messages": [HumanMessage(content="Who approves the OHS regulations?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Who approves the OHS regulations?
================================== Ai Message ==================================
Tool Calls:
  search_documents (call_N9P1qrLdmDrB3w53zj95UHft)
 Call ID: call_N9P1qrLdmDrB3w53zj95UHft
  Args:
    query: OHS regulations approval
================================= Tool Message =================================
Name: search_documents

Frequently Asked 
Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 1 
 1. How will the OHS Regulations affect my business?
The Regulations will impact you 
and your business on various 
levels.
With fewer approvals required 
from the Chief Safety Officer, the 
Regulations strongly support an 
Internal Responsibility System (IRS).
 2. How different are the OHS Regulations from the General Safety Regulations?
There are many changes 
throughout the Regulations. 
The document is easier to use with 
an indexed search feature to help 
you fin

---
## Metadata Filtering

Restrict retrieval to specific pages, sources, or date ranges.

Real-world use cases:
- "Search only in legal documents from 2024"
- "Find info from the methodology section only"
- "Retrieve from department X's knowledge base"

In [20]:
# ──────────────────────────────────────────────────────────
# UNFILTERED SEARCH (baseline)
# ──────────────────────────────────────────────────────────
print("=" * 60)
print("UNFILTERED: Search across all chunks")
print("=" * 60)

query = "How will the OHS Regulations affect my business?"
results = vectorstore.similarity_search(query, k=4)

print(f"\nQuery: '{query}'  |  Results: {len(results)}\n")
for doc in results:
    page = doc.metadata.get("page", "?")
    print(f"  Page {page}: {doc.page_content[:100]}...")

UNFILTERED: Search across all chunks

Query: 'How will the OHS Regulations affect my business?'  |  Results: 4

  Page 0: Frequently Asked 
Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 1 
 1. How will the OHS ...
  Page 1: Frequently Asked Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 2 
  4. How much will it ...
  Page 1: regulations have Codes. The WSCC 
will update Codes to reflect the 
Regulations, but it is important...
  Page 2: Frequently Asked Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 3 
Ask Us. 
It is importa...


In [21]:
# ──────────────────────────────────────────────────────────
# FILTERED: Single page
# ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FILTERED: Only page 1 (abstract and introduction)")
print("=" * 60)

page1_results = vectorstore.similarity_search(
    query="How will the OHS Regulations affect my business?",
    k=4,
    filter={"page": 0},  # Page indexing starts at 0
)

print(f"\nQuery: 'How will the OHS Regulations affect my business?'  |  Filter: page 0  |  Results: {len(page1_results)}\n")
for doc in page1_results:
    page = doc.metadata.get("page", "?")
    print(f"  Page {page}: {doc.page_content[:100]}...")


FILTERED: Only page 1 (abstract and introduction)

Query: 'How will the OHS Regulations affect my business?'  |  Filter: page 0  |  Results: 4

  Page 0: Frequently Asked 
Questions
OCCUPATIONAL HEALTH AND SAFETY REGULATIONS
Page 1 
 1. How will the OHS ...
  Page 0: • Robotics
• Radiation
• Explosives
• Forestry and Mill Operations
• Additional protection for elect...
  Page 0: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and...
  Page 0: mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention ...


In [22]:
# ──────────────────────────────────────────────────────────
# SHOW ALL AVAILABLE METADATA
# ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Available metadata keys in your chunks:")
print("=" * 60)

# Get a sample to show metadata structure
sample = vectorstore.similarity_search(query, k=1)
if sample:
    print(f"\n  Metadata keys: {list(sample[0].metadata.keys())}")
    print(f"  Sample:        {sample[0].metadata}")


Available metadata keys in your chunks:

  Metadata keys: ['source', 'producer', 'total_pages', 'creator', 'page_label', 'page', 'creationdate']
  Sample:        {'source': 'FAQ.pdf', 'producer': 'PDFium', 'total_pages': 3, 'creator': 'PDFium', 'page_label': '1', 'page': 0, 'creationdate': 'D:20260410122842'}


---
## Module 4: What's Next?

| Pattern | Description |
|---------|------------|
| **Agentic RAG** | LLM decides when/what to retrieve, can iterate |
| **Graph RAG** | Knowledge graphs for relationship-aware retrieval |
| **Multimodal RAG** | Images, tables, charts as retrievable content |
| **Hybrid Search** | BM25 (keyword) + semantic search combined |
| **Re-ranking** | Cross-encoder second-pass for precision |

### Key Takeaways

1. **RAG is the practical path** to private-data LLM applications
2. **Retrieval is the bottleneck** — invest in hybrid search + re-ranking
3. **Always evaluate** — use RAGAS from day one
4. **Agentic RAG is the future** — static pipelines have ceilings
5. **You can start today** — LangChain + OpenAI + ChromaDB + a PDF = working RAG